# Lie superalgebras

`lie_superalgebras.py` constructs the **basic classical Lie superalgebras**: the simple Lie superalgebras with an even non-degenerate invariant form, which are the input of affine vertex superalgebras and of quantum Hamiltonian reduction. Sage has no Lie superalgebras, so the module also provides a category for them.

**The construction.** The contragredient Lie superalgebra $\mathfrak g(A, \tau)$ of a Cartan matrix $A$ with a set $\tau$ of odd simple roots (a super Dynkin diagram; Kac 1977) is generated by $e_i, f_i, h_i$, with $e_i, f_i$ odd for $i \in \tau$, subject to
$$[e_i, f_j] = \delta_{ij} h_i, \qquad [h_i, e_j] = a_{ij} e_j, \qquad [h_i, f_j] = -a_{ij} f_j, \qquad [h_i, h_j] = 0,$$
modulo the largest ideal that meets the Cartan subalgebra trivially. It is built root space by root space. For a positive root $\beta$ of height at least 2, $\mathfrak g_\beta$ is spanned by the $[e_i, x]$ with $x \in \mathfrak g_{\beta - \alpha_i}$, and such an element is zero exactly when every $\operatorname{ad} f_j$ kills it. So no Serre relations are needed: the extra relations at odd isotropic nodes come out by themselves. The invariant form is found by solving the invariance equations. A singular Cartan matrix, as for $\mathfrak{psl}(m|m)$, is quotiented by the centre.

**Named types.** These use the distinguished Dynkin diagram (Frappat–Sciarrino–Sorba, *Dictionary on Lie superalgebras*, arXiv:hep-th/9607161). The form is normalised as in the tables of Kac–Wakimoto (*Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004)): $(\theta|\theta) = 2$ for $\theta$ the highest root of a **chosen** simple component of the even part. The choice is part of the name:

| name | $\theta$ in | $h^\vee$ |
|---|---|---|
| `sl(m|n)` | $\mathfrak{sl}_m$ ($m \ge 2$, $m \ne n$) | $m - n$ |
| `psl(m|m)` | $\mathfrak{sl}_m$ ($m \ge 2$) | $0$ |
| `osp(m|n)` | $\mathfrak{so}_m$ ($m \ge 4$) | $m - n - 2$ |
| `spo(n|m)` | $\mathfrak{sp}_n$ ($n \ge 2$) | $(n - m)/2 + 1$ |
| `D(2,1;a)` | one $\mathfrak{sl}_2$ | $0$ |
| `F(4)` | $\mathfrak{sl}_2$ / $\mathfrak{so}_7$ (`theta=`) | $-2$ / $3$ |
| `G(3)` | $\mathfrak{sl}_2$ / $G_2$ (`theta=`) | $-3/2$ / $2$ |

Here `osp(m|n)` and `spo(n|m)` are the same Lie superalgebra with different normalisations, and $n$ is even. Any other Dynkin diagram, or a Lie algebra, can be given by its Cartan matrix.

Contents:

1. A first example: $\mathfrak{sl}(2|1)$
2. The named types and Kac–Wakimoto's normalisation
3. Roots
4. The invariant form and the Killing form
5. Structure constants and the category
6. $D(2,1;a)$ with a symbolic parameter
7. Other Dynkin diagrams, Lie algebras, refusals
8. Affine vertex superalgebras
9. Unique representation and pickling

In [1]:
from vertex_algebras.lie_superalgebras import LieSuperalgebra, LieSuperalgebras
from vertex_algebras.affine_vertex_algebra import UniversalAffineVertexAlgebra


def short(G):
    "the name of a Lie superalgebra without the base ring"
    return str(G).split(' over ')[0].replace('Lie superalgebra ', '')

## 1. A first example: $\mathfrak{sl}(2|1)$

The distinguished diagram of $\mathfrak{sl}(2|1)$ has an even simple root $\alpha_1$ and an odd isotropic simple root $\alpha_2$; the row of an odd isotropic root has a zero diagonal entry. The basis vectors are named as Sage prints Chevalley bases: `E[root]` for root vectors, `h1`, `h2` for the Cartan subalgebra.

In [2]:
g = LieSuperalgebra(QQ, 'sl(2|1)')
print(g)
print('dimension', g.dimension(), '  superdimension', g.superdimension(), '  rank', g.rank())
print('Cartan matrix:')
print(g.cartan_matrix())
print('odd simple roots:', g.odd_simple_roots())
list(g.basis().keys())

Lie superalgebra sl(2|1) over Rational Field
dimension 8   superdimension 0   rank 2
Cartan matrix:
[ 2 -1]
[-1  0]
odd simple roots: (2,)


['E[alpha[1]]',
 'E[alpha[2]]',
 'E[alpha[1] + alpha[2]]',
 'h1',
 'h2',
 'E[-alpha[1]]',
 'E[-alpha[2]]',
 'E[-alpha[1] - alpha[2]]']

Elements are linear combinations of the basis. The bracket is super-antisymmetric: for odd $x, y$, $[x, y] = [y, x]$. Here `h(i)` is $[e_i, f_i]$, and `is_even_odd()` gives the parity.

In [3]:
e1, e2, f1, f2 = g.e(1), g.e(2), g.f(1), g.f(2)
print('parities:', [x.is_even_odd() for x in (e1, e2, f1, f2)])
print('[e2, f2] =', e2.bracket(f2), '   [f2, e2] =', f2.bracket(e2), '   [e2, e2] =', e2.bracket(e2))
print('[e1, e2] =', e1.bracket(e2), '   [h1, e2] =', g.h(1).bracket(e2))
x = e1 + 2*g.h(2) - 3*f2
x.bracket(e2)

parities: [0, 1, 0, 1]
[e2, f2] = h2    [f2, e2] = h2    [e2, e2] = 0
[e1, e2] = -E[alpha[1] + alpha[2]]    [h1, e2] = -E[alpha[2]]


-E[alpha[1] + alpha[2]] - 3*h2

## 2. The named types and Kac–Wakimoto's normalisation

Dimension, superdimension, $h^\vee$, the normalisation root $\theta$ (in simple-root coordinates) and the simple components of the even part. The values of $h^\vee$ are those of Kac–Wakimoto's Tables I–III. They are computed from the Killing form, $\kappa = 2h^\vee (\cdot|\cdot)$, not looked up.

In [4]:
Ka.<a> = Frac(QQ['a'])
algebras = [('sl(3|2)', LieSuperalgebra(QQ, 'sl(3|2)')), ('psl(2|2)', LieSuperalgebra(QQ, 'psl(2|2)')),
            ('osp(5|2)', LieSuperalgebra(QQ, 'osp(5|2)')), ('osp(4|2)', LieSuperalgebra(QQ, 'osp(4|2)')),
            ('spo(2|3)', LieSuperalgebra(QQ, 'spo(2|3)')), ('spo(4|1)', LieSuperalgebra(QQ, 'spo(4|1)')),
            ('D(2,1;a)', LieSuperalgebra(Ka, 'D(2,1;a)', a=a)),
            ('F(4), θ in sl_2', LieSuperalgebra(QQ, 'F(4)', theta='sl2')),
            ('F(4), θ in so_7', LieSuperalgebra(QQ, 'F(4)', theta='so7')),
            ('G(3), θ in sl_2', LieSuperalgebra(QQ, 'G(3)', theta='sl2')),
            ('G(3), θ in G_2', LieSuperalgebra(QQ, 'G(3)', theta='G2'))]
print('%-17s %4s %5s %6s  %-14s %s' % ('', 'dim', 'sdim', 'h^v', 'θ', 'even part'))
for name, G in algebras:
    print('%-17s %4d %5d %6s  %-14s %s' % (name, G.dimension(), G.superdimension(), G.dual_coxeter_number(),
                                       G.theta(), ' + '.join(str(ct) for ct, _, _ in G.even_part_components())))

                   dim  sdim    h^v  θ              even part
sl(3|2)             24     0      1  (1, 1, 0, 0)   ['A', 2] + ['A', 1]
psl(2|2)            14    -2      0  (1, 0, 0)      ['A', 1] + ['A', 1]
osp(5|2)            23     3      1  (0, 1, 2)      ['B', 2] + ['A', 1]
osp(4|2)            17     1      0  (0, 0, 1)      ['A', 1] + ['A', 1] + ['A', 1]
spo(2|3)            12     0    1/2  (2, 2)         ['A', 1] + ['A', 1]
spo(4|1)            14     6    5/2  (2, 2)         ['B', 2]
D(2,1;a)            17     1      0  (2, 1, 1)      ['A', 1] + ['A', 1] + ['A', 1]
F(4), θ in sl_2     40     8     -2  (2, 3, 2, 1)   ['B', 3] + ['A', 1]
F(4), θ in so_7     40     8      3  (0, 2, 2, 1)   ['B', 3] + ['A', 1]
G(3), θ in sl_2     31     3   -3/2  (2, 4, 2)      ['G', 2] + ['A', 1]
G(3), θ in G_2      31     3      2  (0, 3, 2)      ['G', 2] + ['A', 1]


The two normalisations of $F(4)$. With $\theta$ in $\mathfrak{sl}_2$ the $\mathfrak{sl}_2$ root has norm 2; with $\theta$ in $\mathfrak{so}_7$ the highest root of $\mathfrak{so}_7$ does. The forms differ by the factor $-2/3$, and $h^\vee$ by its inverse: $-2 = -\tfrac23 \cdot 3$. The reduction for $\theta$ in $\mathfrak{sl}_2$ gives the $N = 8$ superconformal algebra ($\mathfrak g^\natural = \mathfrak{so}_7$).

In [5]:
for th in ('sl2', 'so7'):
    F = LieSuperalgebra(QQ, 'F(4)', theta=th)
    print('theta in %-4s h^v = %2s   highest roots of the components and their norms: %s'
          % (th, F.dual_coxeter_number(), [(str(ct), t, F.root_norm(t)) for ct, t, _ in F.even_part_components()]))

theta in sl2  h^v = -2   highest roots of the components and their norms: [("['B', 3]", (0, 2, 2, 1), -4/3), ("['A', 1]", (2, 3, 2, 1), 2)]
theta in so7  h^v =  3   highest roots of the components and their norms: [("['B', 3]", (0, 2, 2, 1), 2), ("['A', 1]", (2, 3, 2, 1), -3)]


## 3. Roots

Roots are given in simple-root coordinates; `root_norm` is the normalised form on the roots. Odd roots can be isotropic, like the odd roots of $\mathfrak{osp}(5|2)$ of norm 0.

In [6]:
G = LieSuperalgebra(QQ, 'osp(5|2)')
print('Cartan matrix:'); print(G.cartan_matrix())
print('odd simple roots:', G.odd_simple_roots(), '  theta:', G.theta())
for b in G.positive_roots():
    print('%-12s %-5s norm %s' % (b, 'odd' if b in G.odd_roots() else 'even', G.root_norm(b)))

Cartan matrix:
[ 0 -1  0]
[-1  2 -1]
[ 0 -2  2]
odd simple roots: (1,)   theta: (0, 1, 2)
(1, 0, 0)    odd   norm 0
(0, 1, 0)    even  norm 2
(0, 0, 1)    even  norm 1
(1, 1, 0)    odd   norm 0
(0, 1, 1)    even  norm 1
(1, 1, 1)    odd   norm -1
(0, 1, 2)    even  norm 2
(1, 1, 2)    odd   norm 0
(1, 2, 2)    odd   norm 0
(2, 2, 2)    even  norm -4


An odd **non-isotropic** simple root, as in $\mathfrak{osp}(1|2)$ = `spo(2|1)`: $[e, e]$ is the root vector of the even root $2\alpha$. Its name is `E[2*alpha[1]]`; written as `2*alpha[1]` it would be indistinguishable from twice `alpha[1]`.

In [7]:
O = LieSuperalgebra(QQ, 'spo(2|1)')
e, f = O.e(1), O.f(1)
print('roots', O.roots(), '  norms', O.root_norm((1,)), O.root_norm((2,)), '  h^v', O.dual_coxeter_number())
print('[e, e] =', e.bracket(e), '   root of E[2*alpha[1]]:', O.root_on_basis('E[2*alpha[1]]'))
print('root vector of 2 alpha:', O.root_vector((2,)), '  of -alpha:', O.root_vector((1,), -1))

roots [(1,), (2,), (-1,), (-2,)]   norms 1/2 2   h^v 3/2
[e, e] = E[2*alpha[1]]    root of E[2*alpha[1]]: (2,)
root vector of 2 alpha: E[2*alpha[1]]   of -alpha: E[-alpha[1]]


## 4. The invariant form and the Killing form

`invariant_form_matrix()` is the Gram matrix of the normalised form in the basis order, and `invariant_form(x, y)` evaluates it. The form is supersymmetric: $(x|y) = (-1)^{|x||y|}(y|x)$. The super Killing form $\operatorname{str}(\operatorname{ad} x \operatorname{ad} y)$ is $2h^\vee$ times it.

In [8]:
g = LieSuperalgebra(QQ, 'sl(2|1)')
print(g.invariant_form_matrix())
print('(e1|f1) =', g.invariant_form(g.e(1), g.f(1)), '  (e2|f2) =', g.invariant_form(g.e(2), g.f(2)),
      '  (f2|e2) =', g.invariant_form(g.f(2), g.e(2)))
B = g.basis()
all(g.killing_form(x, y) == 2 * g.dual_coxeter_number() * g.invariant_form(x, y) for x in B for y in B)

[ 0  0  0  0  0  1  0  0]
[ 0  0  0  0  0  0  1  0]
[ 0  0  0  0  0  0  0 -1]
[ 0  0  0  2 -1  0  0  0]
[ 0  0  0 -1  0  0  0  0]
[ 1  0  0  0  0  0  0  0]
[ 0 -1  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0]
(e1|f1) = 1   (e2|f2) = 1   (f2|e2) = -1


True

For $\mathfrak{psl}(2|2)$, $h^\vee = 0$: the Killing form vanishes identically, but the invariant form is non-degenerate.

In [9]:
P = LieSuperalgebra(QQ, 'psl(2|2)')
print(P.dimension(), P.superdimension(), P.dual_coxeter_number())
print(all(P.killing_form(x, y) == 0 for x in P.basis() for y in P.basis()), P.invariant_form_matrix().det() != 0)

14 -2 0
True True


## 5. Structure constants and the category

`structure_coefficients()` lists all nonzero brackets of basis vectors, the format the affine constructor takes for structure constants.

In [10]:
O.structure_coefficients()

{('E[alpha[1]]', 'E[alpha[1]]'): {'E[2*alpha[1]]': 1},
 ('E[alpha[1]]', 'h1'): {'E[alpha[1]]': -2},
 ('E[alpha[1]]', 'E[-alpha[1]]'): {'h1': 1},
 ('E[alpha[1]]', 'E[-2*alpha[1]]'): {'E[-alpha[1]]': -4},
 ('E[2*alpha[1]]', 'h1'): {'E[2*alpha[1]]': -4},
 ('E[2*alpha[1]]', 'E[-alpha[1]]'): {'E[alpha[1]]': -4},
 ('E[2*alpha[1]]', 'E[-2*alpha[1]]'): {'h1': -8},
 ('h1', 'E[alpha[1]]'): {'E[alpha[1]]': 2},
 ('h1', 'E[2*alpha[1]]'): {'E[2*alpha[1]]': 4},
 ('h1', 'E[-alpha[1]]'): {'E[-alpha[1]]': -2},
 ('h1', 'E[-2*alpha[1]]'): {'E[-2*alpha[1]]': -4},
 ('E[-alpha[1]]', 'E[alpha[1]]'): {'h1': 1},
 ('E[-alpha[1]]', 'E[2*alpha[1]]'): {'E[alpha[1]]': 4},
 ('E[-alpha[1]]', 'h1'): {'E[-alpha[1]]': 2},
 ('E[-alpha[1]]', 'E[-alpha[1]]'): {'E[-2*alpha[1]]': 1},
 ('E[-2*alpha[1]]', 'E[alpha[1]]'): {'E[-alpha[1]]': 4},
 ('E[-2*alpha[1]]', 'E[2*alpha[1]]'): {'h1': 8},
 ('E[-2*alpha[1]]', 'h1'): {'E[-2*alpha[1]]': 4}}

Sage has no category of Lie superalgebras: `LieAlgebras(QQ).Super()` is the category of super modules, which forgets the bracket. The tests of Sage's Lie algebras assume antisymmetry, which fails for the brackets of odd elements. The module defines `LieSuperalgebras(R)` over Sage's super modules. Its `TestSuite` checks super antisymmetry, the super Jacobi identity, the parity of brackets and the invariance of the form.

In [11]:
print(LieAlgebras(QQ).Super())
print(LieSuperalgebras(QQ))
print(O.category())
TestSuite(O).run(verbose=True)

Category of super modules over Rational Field
Category of Lie superalgebras over Rational Field
Category of finite dimensional Lie superalgebras with basis over Rational Field
running ._test_additive_associativity() . . .

 pass


running ._test_an_element() . . .

 pass


running ._test_bracket_parity() . . .

 pass


... (45 outputs left out)

running ._test_super_jacobi_identity() . . .

 pass


running ._test_zero() . . .

 pass


## 6. $D(2,1;a)$ with a symbolic parameter

$D(2,1;a)$ is a one-parameter family of 17-dimensional Lie superalgebras with three $\mathfrak{sl}_2$'s in the even part. Its Cartan matrix contains $a$, and so do its brackets and root norms. KW's normalisation puts $\theta$ in one $\mathfrak{sl}_2$ without saying which; here it is the highest root $2\alpha_1 + \alpha_2 + \alpha_3$ of the distinguished positive system, and another can be chosen with `theta=`. Since $h^\vee = 0$, the choice changes only the normalisation of the level.

In [12]:
D = LieSuperalgebra(Ka, 'D(2,1;a)', a=a)
print(D.cartan_matrix())
print('[h1, e3] =', D.h(1).bracket(D.e(3)))
print('sl_2 components (highest root, norm):', [(t, D.root_norm(t)) for _, t, _ in D.even_part_components()])
D3 = LieSuperalgebra(Ka, 'D(2,1;a)', a=a, theta=(0, 1, 0))
print('theta = alpha_2 instead:', [(t, D3.root_norm(t)) for _, t, _ in D3.even_part_components()])

[ 0  1  a]
[-1  2  0]
[-1  0  2]
[h1, e3] = a*E[alpha[3]]
sl_2 components (highest root, norm): [((0, 0, 1), -2*a/(a + 1)), ((0, 1, 0), -2/(a + 1)), ((2, 1, 1), 2)]
theta = alpha_2 instead: [((0, 0, 1), 2*a), ((0, 1, 0), 2), ((2, 1, 1), -2*a - 2)]


$D(2,1;1) \cong \mathfrak{osp}(4|2) = \mathfrak{spo}(2|4)$: the same dimension, $h^\vee$ and even part, $\mathfrak{so}_4 \oplus \mathfrak{sp}_2 = \mathfrak{sl}_2^{\oplus 3}$. The root norms depend on which $\mathfrak{sl}_2$ carries $\theta$:
- the default $\theta$ of $D(2,1;1)$ plays the role of the $\mathfrak{sp}_2$ root, so its norms are those of `spo(2|4)`;
- with $\theta = \alpha_2$, one of the $\mathfrak{so}_4$ roots, they are those of `osp(4|2)`.

In [13]:
D1 = LieSuperalgebra(QQ, 'D(2,1;a)', a=1)
D1b = LieSuperalgebra(QQ, 'D(2,1;a)', a=1, theta=(0, 1, 0))
for name, X in [('D(2,1;1)', D1), ('spo(2|4)', LieSuperalgebra(QQ, 'spo(2|4)')),
                ('D(2,1;1), θ = α_2', D1b), ('osp(4|2)', LieSuperalgebra(QQ, 'osp(4|2)'))]:
    print('%-18s %d %d %s %s  root norms %s' % (name, X.dimension(), X.superdimension(), X.dual_coxeter_number(),
                                              [str(c) for c, _, _ in X.even_part_components()],
                                              sorted(X.root_norm(b) for b in X.positive_roots())))

D(2,1;1)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-1, -1, 0, 0, 0, 0, 2]
spo(2|4)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-1, -1, 0, 0, 0, 0, 2]
D(2,1;1), θ = α_2  17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-4, 0, 0, 0, 0, 2, 2]
osp(4|2)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-4, 0, 0, 0, 0, 2, 2]


## 7. Other Dynkin diagrams, Lie algebras, refusals

A Lie superalgebra has inequivalent Dynkin diagrams, related by odd reflections, and any of them can be given by its Cartan matrix. Two examples:
- $\mathfrak{sl}(2|1)$ with both simple roots odd;
- $\mathfrak{osp}(3|2)$ with simple roots $\varepsilon - \delta$ (odd isotropic) and $\delta$ (odd non-isotropic), against the named `spo(2|3)`.

For a Cartan matrix, `theta=` gives the normalisation root; it is required when the highest root is odd.

In [14]:
H = LieSuperalgebra(QQ, matrix([[0, 1], [1, 0]]), odd=[1, 2], theta=(1, 1))
print(H)
print(H.dimension(), H.superdimension(), H.dual_coxeter_number(), '  as sl(2|1):', g.dimension(), g.superdimension(), g.dual_coxeter_number())
O32 = LieSuperalgebra(QQ, matrix([[0, 1], [-2, 2]]), odd=[1, 2], theta=(0, 2))
S23 = LieSuperalgebra(QQ, 'spo(2|3)')
[(X.dimension(), X.superdimension(), X.dual_coxeter_number(), [str(c) for c, _, _ in X.even_part_components()])
 for X in (O32, S23)]

Lie superalgebra with Cartan matrix [[0, 1], [1, 0]] and odd simple roots [1, 2] over Rational Field
8 0 1   as sl(2|1): 8 0 1


[(12, 0, 1/2, ["['A', 1]", "['A', 1]"]),
 (12, 0, 1/2, ["['A', 1]", "['A', 1]"])]

Without odd simple roots the construction gives the simple Lie algebras, with KW's Table I normalisation, the long roots of norm 2.

In [15]:
for t, r in [('B', 3), ('G', 2), ('F', 4), ('E', 6)]:
    L = LieSuperalgebra(QQ, CartanMatrix([t, r]))
    print('%s%d: dimension %3d, h^v %2d (Sage: %2d)' % (t, r, L.dimension(), L.dual_coxeter_number(),
                                                     CartanType([t, r]).dual_coxeter_number()))

B3: dimension  21, h^v  5 (Sage:  5)
G2: dimension  14, h^v  4 (Sage:  4)


F4: dimension  52, h^v  9 (Sage:  9)


E6: dimension  78, h^v 12 (Sage: 12)


In [16]:
for args, kw in [(('F(4)',), {}), (('osp(3|2)',), {}), (('sl(2|2)',), {}),
                 ((matrix([[2, -2], [-2, 2]]),), {}), ((matrix([[2, -1], [-1, 0]]),), {'odd': [2]})]:
    try:
        LieSuperalgebra(QQ, *args, **kw)
    except ValueError as err:
        print('ValueError:', err)

ValueError: F(4) has two normalisations of the form: give theta='sl2' or theta='so7'
ValueError: osp(m|n) has θ in so_m: m >= 4, n even (and so_4 is not simple); use spo(n|m) for θ in sp_n
ValueError: sl(m|m) is not simple: use psl(m|m)
ValueError: the Lie superalgebra of this Cartan matrix is not finite dimensional (roots of height above 100)
ValueError: the highest root is odd: give theta (a highest root of a simple component of the even part)


## 8. Affine vertex superalgebras

`UniversalAffineVertexAlgebra(R, g, level=k)` takes a Lie superalgebra. It uses the basis as generator names, the normalised form times the level, and the grading by the root lattice. The constructor checks the super Jacobi identity and the invariance of the form, and builds the Sugawara vector, whose central charge is $c = k \operatorname{sdim}\mathfrak g/(k + h^\vee)$.

In [17]:
K.<k> = Frac(QQ['k'])
for G in [LieSuperalgebra(QQ, 'spo(2|1)'), LieSuperalgebra(QQ, 'sl(2|1)'), LieSuperalgebra(QQ, 'psl(2|2)'),
          LieSuperalgebra(QQ, 'osp(4|2)'), LieSuperalgebra(QQ, 'spo(4|1)'),
          LieSuperalgebra(QQ, 'F(4)', theta='sl2'), LieSuperalgebra(QQ, 'G(3)', theta='sl2')]:
    V = UniversalAffineVertexAlgebra(K, G, level=k, conformal_vector='sugawara')
    expected = k * G.superdimension() / (k + G.dual_coxeter_number())
    print('%-10s c = %-14s %s' % (short(G), V.central_charge(), V.central_charge() == expected))

spo(2|1)   c = k/(k + 3/2)    True
sl(2|1)    c = 0              True
psl(2|2)   c = -2             True
osp(4|2)   c = 1              True


spo(4|1)   c = 6*k/(k + 5/2)  True


F(4)       c = 8*k/(k - 2)    True


G(3)       c = 3*k/(k - 3/2)  True


The currents of $V^k(\mathfrak{osp}(1|2))$: $x(z) x(w) \sim [x, x](w)/(z-w)$ and $x(z) y(w) \sim k(x|y)/(z-w)^2 + [x, y](w)/(z-w)$, graded by the root lattice.

In [18]:
V = UniversalAffineVertexAlgebra(K, O, level=k, conformal_vector='sugawara')
print(V)
print(V.variable_names())
x, y = V.gen('E[alpha[1]]'), V.gen('E[-alpha[1]]')
print('x x:', x.ope(x), '   x y:', x.ope(y), '   grade of x:', x.horizontal_grade())
print(V.lie_algebra() is O)

Universal affine vertex superalgebra of Lie superalgebra spo(2|1) over Rational Field at level k over Fraction Field of Univariate Polynomial Ring in k over Rational Field
('E[alpha[1]]', 'E[2*alpha[1]]', 'h1', 'E[-alpha[1]]', 'E[-2*alpha[1]]')
x x: [E[2*alpha[1]]]    x y: [h1, 4*k]    grade of x: (1)
True


The character of the universal affine vertex superalgebra is $\prod_{n \ge 1} (1+q^n)^{\dim \mathfrak g_{\bar 1}} / (1-q^n)^{\dim \mathfrak g_{\bar 0}}$; for $\mathfrak{osp}(1|2)$ that is $(1+q^n)^2/(1-q^n)^3$.

In [19]:
ch = V.character(8)
R.<q> = PowerSeriesRing(QQ, default_prec=8)
expected = prod((1 + q^n)^2 / (1 - q^n)^3 for n in range(1, 8))
print(ch)
L = ch.laurent_part()
print(all(L[n] == expected[n] for n in range(8)))

1 + 5*q + 18*q^2 + 55*q^3 + 149*q^4 + 371*q^5 + 867*q^6 + 1923*q^7 + O(q^8)
True


$D(2,1;a)$ over $\mathbb Q(a, k)$: $\operatorname{sdim} = 1$ and $h^\vee = 0$, so $c = 1$ for every $a$ and $k$.

In [20]:
Kak.<aa, kk> = Frac(QQ['aa', 'kk'])
Da = LieSuperalgebra(Frac(QQ['aa']), 'D(2,1;a)', a=Frac(QQ['aa']).gen())
UniversalAffineVertexAlgebra(Kak, Da, level=kk, conformal_vector='sugawara').central_charge()

1

## 9. Unique representation and pickling

Equal data give the same object, and pickles come back as the same object.

In [21]:
print(g is LieSuperalgebra(QQ, 'sl(2|1)'), loads(dumps(g)) is g, loads(dumps(D)) is D, loads(dumps(H)) is H)
print(loads(dumps(V)) is V)

True True True True
True
